# Conclusion and reproducible delivery — FD001

[Português — canonical edition](../pt-BR/27_conclusao_entrega_fd001.ipynb) · [English README](../../README.en.md)

The FD001 stage was completed with an Extra Trees model, optimized using engine-level validation and trained on causal temporal features. The input representation contains 205 features; 93 were selected by the final pipeline. This English edition follows the canonical Portuguese notebook. Reusable logic is maintained in `src`, and predictions can be produced through the delivery scripts.

## Dataset provenance and the meaning of NASA

The project uses the **Turbofan Engine Degradation Simulation Data Set**, provided by the Prognostics Center of Excellence (PCoE) at NASA Ames Research Center (Saxena and Goebel, 2008). NASA stands for **National Aeronautics and Space Administration**, the United States agency for space and aeronautics. In this project's title, it identifies the data source; the analysis and model are independent portfolio work. [Official NASA repository](https://www.nasa.gov/intelligent-systems-division/discovery-and-systems-health/pcoe/pcoe-data-set-repository/).

The time series were generated with **C-MAPSS — Commercial Modular Aero-Propulsion System Simulation**, which models the operation and degradation of commercial aircraft turbofan engines. These are simulated data: training trajectories continue until failure, while test histories stop earlier and the true RUL at the last cycle is supplied separately. [Official dataset catalog](https://data.nasa.gov/dataset/cmapss-jet-engine-simulated-data).

This project uses **FD001**, with 100 training engines and 100 test engines, one operating condition and one failure mode: high-pressure compressor (HPC) degradation. This controlled setting supports remaining-life prediction research; it does not demonstrate performance on a real fleet.

**Dataset reference:** Saxena, A.; Goebel, K. (2008). *Turbofan Engine Degradation Simulation Data Set*. NASA Ames, Prognostics Data Repository. [Original source and credit](https://www.nasa.gov/intelligent-systems-division/discovery-and-systems-health/pcoe/pcoe-data-set-repository/).

Insights: Attribution identifies the data providers and distinguishes the benchmark's origin from authorship of this analysis. Because FD001 is simulated, results must be interpreted within that setting.

## Benchmark results

| Model | MAE (cycles) | RMSE (cycles) | Mean signed error (cycles) |
| --- | ---: | ---: | ---: |
| Extra Trees | 16.23 | 23.80 | +10.08 |
| XGBoost | 17.29 | 25.15 | +9.31 |
| HistGradientBoosting | 17.77 | 27.01 | +8.83 |

Insights: All candidates were evaluated on the same 100 official test engines after their hyperparameters had been frozen. Extra Trees was selected after this comparison and achieved the lowest MAE and RMSE. This benchmark is not an additional independent evaluation after selecting the winner. The mean overestimation of 10.08 cycles should be considered when interpreting maintenance risk. These values were transcribed from the frozen record in notebook 21; no new test evaluation was performed for this delivery.

## Feature representation and explainability

Rolling means and standard deviations were calculated over 5, 10, 20, 40 and 80 cycles, separately for each engine and using only measurements available at or before the prediction cycle. Deltas were offered to the selector, but none were retained in the final fit. The selected set contains 13 base features, 60 rolling means and 20 rolling standard deviations.

Insights: The observed cycle was the feature with the highest permutation importance; standard deviations over longer windows also made a relevant contribution. These findings describe predictive dependence on sensor history and do not establish physical causality. Correlated features can share importance.

## Operational usefulness and limitations

An illustrative predicted-RUL threshold of 20 cycles produced timely alerts across the 100 complete OOF trajectories, with three-cycle confirmation and a useful lead-time window of 5 to 30 cycles. Median lead time was 16 cycles. Cost scenarios were hypothetical and analyzed only on development data.

Insights: This result demonstrates how a prediction can support a maintenance policy, but does not validate deployment. The folds were used during optimization, and the official test contains truncated trajectories. Real operation requires costs and minimum lead times defined by the maintenance team.

In the exploratory uncertainty analysis, mean within-engine cycle coverage was 85.9%, mean interval width was 91.3 cycles, and 46% of trajectories were fully covered. For predicted RUL above 100 cycles, mean coverage was 76.4%.

Insights: The global interval was wide and did not reach the calibration reference in every regime. The delivery provides point predictions only; OOF intervals are not presented as validated intervals for the model refitted on all training data.

## Delivery and reproduction

The frozen pipeline is available at `../../outputs/models/extra_trees_fd001_v1/model.joblib`. Its companion `metadata.json` records parameters, features, hashes, versions and checks. Observed history is supplied as a CSV and processed by `../../scripts/predict_final_extra_trees_fd001.py`. Continuous history starting at cycle 1 is required, including identifiers, cycle and 17 signals, as described in the [English README](../../README.en.md).

The included example uses a prefix of the training data and demonstrates the interface. Delivery checks compare predictions before and after serialization, verify feature construction, confirm that future cycles do not change earlier predictions, and reject duplicate cycles or incomplete histories.

Insights: The package enables local predictions without rerunning Optuna. These checks validate pipeline behavior and the temporal input contract; they do not provide another estimate of out-of-sample performance.

## Publication reading path

The main reading path starts with this summary, followed by temporal ablation, screening, optimization, the benchmark, model selection and error analysis, explainability, alerts and uncertainty. Other notebooks are preserved as experiment history; readers do not need to follow all 27 notebooks in sequence.

Insights: The main narrative has been separated from the experiment history to make the portfolio easier to assess. The README documents this organization, and no experiment was removed. The README, HTML report and notebook 27 are available in both languages; the other notebooks remain in canonical Portuguese. External publication and expansion to FD002–FD004 remain future work.